# 6. 多MCP服务与错误处理

第三节编写了计算服务，第五节编写了套餐服务。实际的 Agent 常常同时连接多个 MCP 服务，还会保留一些本地工具。多个服务的工具放在一起，名称会不会冲突？某个服务连不上时，失败出现在哪里，Agent 会不会中断？

本节用 **MCP 配置字典**同时接入这两个服务，再加入一个本地工具，交给同一个 Agent。然后让套餐服务在不同时机停止，观察失败的表现，并用第八章的中间件处理连接异常。

学习目标：

1. 用 MCP 配置字典接入两个服务，说明工具名前缀从哪里来。
2. 把 MCP 工具与本地工具交给同一个 Agent，从消息记录区分工具来源。
3. 观察服务在发现工具时连不上、执行工具时断开，失败分别以什么形式出现。
4. 用 `ToolRetryMiddleware` 把连接异常转成错误消息，让 Agent 继续回答用户。

本节会请求聊天模型：6.4 和 6.6 共运行三次 Agent，沿用第一节的 `ChatDeepSeek` 配置；其余实验不请求模型。HTTP 服务只监听 `127.0.0.1`，由实验自己启动和关闭。选择项目 `.venv` 内核后，按顺序执行。

## 6.1 本节的服务与工具

![配置键成为工具名前缀](assets/18-mcp-config-multi-server.svg)

| 来源 | 提供的工具 | 本节的用途 |
|---|---|---|
| 计算服务 [calculator_server.py](servers/calculator_server.py) | `add`、`divide` | 加法与除法 |
| 套餐服务 [plan_server.py](servers/plan_server.py) | `get_plan` | 查询套餐的价格和额度 |
| 本地 `@tool` | `multiply` | 计算服务没有乘法，用本地工具补上 |

参数错误与业务错误已在第三节 3.5 验证过：它们由服务端报告，Agent 会收到 `status="error"` 的 ToolMessage。本节只观察**服务连不上**这一类失败。

## 6.2 环境准备

除了 FastMCP 与 MCPAdapter，本节还会用到第七章的 `create_agent`、第八章的 `ToolRetryMiddleware`，以及第一节的模型配置。

多服务组合时，FastMCP 会为每个服务打印一条 INFO 提示：组合服务会复用已经建立的会话处理所有请求。本节的工具调用不依赖请求上下文，所以把这个日志器调到 WARNING，避免输出被重复提示占满；WARNING 及以上的日志仍会显示。

In [19]:
import asyncio
import json
import logging
import os
import socket
import sys
from contextlib import asynccontextmanager, contextmanager
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv
from fastmcp import Client
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.mcp import MCPAdapter
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 lang-chain-learn 项目或其子目录运行本笔记。")

CALC_SCRIPT = PROJECT_ROOT / "Chapter11_MCP" / "servers" / "calculator_server.py"
PLAN_SCRIPT = PROJECT_ROOT / "Chapter11_MCP" / "servers" / "plan_server.py"
ENV_FILE = PROJECT_ROOT / "conf" / ".env"
load_dotenv(dotenv_path=ENV_FILE, override=True)

logging.getLogger("fastmcp.server.providers.proxy").setLevel(logging.WARNING)

print("Python:", sys.executable)
for package in ["langchain", "fastmcp", "langchain-deepseek"]:
    print(f"{package}: {version(package)}")
print("计算服务脚本存在:", CALC_SCRIPT.is_file())
print("套餐服务脚本存在:", PLAN_SCRIPT.is_file())

Python: /Users/skk/Developer/lang-chain-learn/.venv/bin/python
langchain: 1.4.3
fastmcp: 4.1.0
langchain-deepseek: 1.1.1
计算服务脚本存在: True
套餐服务脚本存在: True


模型配置与第一节相同：`deepseek-v4-flash`，关闭思考模式。初始化模型对象还不会发送请求。

In [20]:
if not os.getenv("DEEPSEEK_API_KEY"):
    raise ValueError(f"请在 {ENV_FILE} 中配置 DEEPSEEK_API_KEY，再运行 Agent 部分。")

MODEL_NAME = "deepseek-v4-flash"
model = ChatDeepSeek(
    model=MODEL_NAME,
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    timeout=60,
    max_retries=1,
)
print("模型:", type(model).__name__, MODEL_NAME)

模型: ChatDeepSeek deepseek-v4-flash


## 6.3 用MCP配置字典接入两个服务

### 6.3.1 配置字典的结构

第一节 1.4.1 提到，`MCPAdapter` 可以接收 `{"mcpServers": {...}}` 形式的配置字典。`mcpServers` 下的**每个键是一个服务的名称**，值说明怎样连接它。连接方式由字段推断：有 `command` 是 stdio，有 `url` 是 HTTP。

| 连接方式 | 常用字段 | 含义 |
|---|---|---|
| stdio | `command`、`args` | 启动服务的程序与参数，相当于第四节的 `python 脚本路径` |
| | `cwd`、`env` | 子进程的工作目录与额外环境变量 |
| HTTP | `url` | 完整的 MCP URL，例如 `http://127.0.0.1:8001/mcp` |
| | `headers`、`auth` | 请求头与认证配置，本节不用 |

两个服务都用 stdio 启动；`command` 使用当前内核的 Python，与第四节一致。

In [21]:
def stdio_server(script: Path) -> dict:
    """生成一个 stdio 服务的配置：用当前内核的 Python 启动脚本。"""
    return {"command": sys.executable, "args": [str(script)], "cwd": str(PROJECT_ROOT)}

# 实际 key 是MCP提供的某种功能、工具
# value 连接方式
mcp_config = {
    "mcpServers": {
        "calc": stdio_server(CALC_SCRIPT),
        "plans": stdio_server(PLAN_SCRIPT),
    }
}
# HTTP的写法
# "calc": {
#     "url": "http://127.0.0.1:8000/mcp",
#     "headers": {"Authorization": f"Bearer {os.environ['MCP_TOKEN']}"},
# }

# 替换到配置字典
# mcp_config = {
#     "mcpServers": {
#         "calc": http_server("http://127.0.0.1:8000/mcp"),
#         "plans": {"url": "http://127.0.0.1:8001/mcp"},   # 直接写也行
#     }
# }

for name, server in mcp_config["mcpServers"].items():
    print(f"{name}: 启动 {Path(server['args'][0]).relative_to(PROJECT_ROOT)}")

calc: 启动 Chapter11_MCP/servers/calculator_server.py
plans: 启动 Chapter11_MCP/servers/plan_server.py


### 6.3.2 发现工具：配置键成为名称前缀

把配置字典交给 `MCPAdapter`。进入上下文时，FastMCP 在**客户端进程内**创建一个名为 `MCPRouter` 的组合服务，按配置键挂载两个服务，再像单个服务一样发现工具。

In [22]:
from rich import print as rprint
async with MCPAdapter(mcp_config) as adapter:
    mcp_tools = await adapter.list_tools()

# 工具名从原始名变成了 配置键_原始名
for item in mcp_tools:
    rprint(item)
    print("======"*10)
    server_name = item.metadata["mcp"]["server"]["name"]
    print(f"{item.name:<16} 服务: {server_name:<10} 描述: {item.description}")
    print("======"*10)
    rprint(server)
    print("======"*10)

StructuredTool(
    name='calc_add',
    description='计算两个整数之和。',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}},
        'required': ['a', 'b']
    },
    metadata={
        'mcp': {
            'tool': {'_meta': {'fastmcp': {'tags': []}}},
            'server': {'name': 'MCPRouter', 'version': '4.1.0'}
        }
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x11759a700>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x11cfa1bc0>
)

calc_add         服务: MCPRouter  描述: 计算两个整数之和。


{
    'command': '/Users/skk/Developer/lang-chain-learn/.venv/bin/python',
    'args': ['/Users/skk/Developer/lang-chain-learn/Chapter11_MCP/servers/plan_server.py'],
    'cwd': '/Users/skk/Developer/lang-chain-learn'
}

StructuredTool(
    name='calc_divide',
    description='计算分子除以分母，并按指定精度四舍五入。',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'numerator': {'type': 'number'},
            'denominator': {'type': 'number'},
            'precision': {
                'default': 2,
                'description': '保留的小数位数，范围为 0 到 6。',
                'maximum': 6,
                'minimum': 0,
                'type': 'integer'
            }
        },
        'required': ['numerator', 'denominator']
    },
    metadata={
        'mcp': {
            'tool': {'_meta': {'fastmcp': {'tags': []}}},
            'server': {'name': 'MCPRouter', 'version': '4.1.0'}
        }
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x11759a700>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x11cfa1da0>
)

calc_divide      服务: MCPRouter  描述: 计算分子除以分母，并按指定精度四舍五入。


{
    'command': '/Users/skk/Developer/lang-chain-learn/.venv/bin/python',
    'args': ['/Users/skk/Developer/lang-chain-learn/Chapter11_MCP/servers/plan_server.py'],
    'cwd': '/Users/skk/Developer/lang-chain-learn'
}

StructuredTool(
    name='plans_get_plan',
    description='查询一个套餐的价格、成员人数上限、月度 AI 问答额度和 API 调用额度。',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {'plan': {'enum': ['试用版', '基础版', '专业版', '企业版'], 'type': 'string'}},
        'required': ['plan']
    },
    metadata={
        'mcp': {
            'tool': {'_meta': {'fastmcp': {'tags': []}}},
            'server': {'name': 'MCPRouter', 'version': '4.1.0'}
        }
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x11759a700>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x11cfa2ca0>
)

plans_get_plan   服务: MCPRouter  描述: 查询一个套餐的价格、成员人数上限、月度 AI 问答额度和 API 调用额度。


{
    'command': '/Users/skk/Developer/lang-chain-learn/.venv/bin/python',
    'args': ['/Users/skk/Developer/lang-chain-learn/Chapter11_MCP/servers/plan_server.py'],
    'cwd': '/Users/skk/Developer/lang-chain-learn'
}

工具名都加上了**配置键 + 下划线**作为前缀，描述保持不变。适配后的元数据里，服务名统一是组合服务 `MCPRouter`，所以只能从前缀看出工具来自哪个服务。

前缀同样作用于第五节的 Resource 和 Prompt。用 FastMCP Client 连接同一个配置字典查看：

In [23]:
async with Client(mcp_config) as config_client:
    config_resources = await config_client.list_resources()
    config_prompts = await config_client.list_prompts()

rprint(config_prompts)
rprint(config_resources)
print("资料 URI:", [str(item.uri) for item in config_resources])
print("模板名称:", [item.name for item in config_prompts])

[
    Prompt(
        name='plans_plan_answer',
        title=None,
        description='按客服口径回答套餐问题的提示词模板。',
        arguments=[
            PromptArgument(name='question', title=None, description=None, required=True),
            PromptArgument(name='style', title=None, description=None, required=False)
        ],
        icons=None,
        meta={'fastmcp': {'tags': []}}
    )
]

[
    Resource(
        name='plan_catalog',
        title=None,
        uri='plans://plans/catalog',
        description='四档标准订阅套餐的对比表。',
        mime_type='text/markdown',
        size=None,
        icons=None,
        annotations=None,
        meta={'fastmcp': {'tags': []}}
    )
]

资料 URI: ['plans://plans/catalog']
模板名称: ['plans_plan_answer']


| 能力 | 原服务中的名称 | 组合后的名称 |
|---|---|---|
| Tool | `add`、`divide`、`get_plan` | `calc_add`、`calc_divide`、`plans_get_plan` |
| Resource | `plans://catalog` | `plans://plans/catalog`，前缀加在 URI 路径的开头 |
| Prompt | `plan_answer` | `plans_plan_answer` |

两个服务即使都有名为 `add` 的工具，加上不同前缀后也不会冲突。

### 6.3.3 修改一个条件：换一个配置键

把计算服务的键从 `calc` 改成 `calculator`，服务脚本不变。先预测新的工具名。

In [24]:
renamed_config = {
    "mcpServers": {
        "calculator": stdio_server(CALC_SCRIPT),
        "plans": stdio_server(PLAN_SCRIPT),
    }
}

async with MCPAdapter(renamed_config) as adapter:
    renamed_tools = await adapter.list_tools()

print("改名后的工具:", [item.name for item in renamed_tools])

改名后的工具: ['calculator_add', 'calculator_divide', 'plans_get_plan']


前缀来自给给适配器赋值的配置字典的key，而不是服务脚本里的 `FastMCP(name="Calculator")`。改配置键就会改工具名；如果 Agent 的系统提示或中间件写了工具名，也要一起修改。

## 6.4 本地工具与MCP工具一起交给Agent

### 6.4.1 定义本地工具并合并工具列表

计算服务没有乘法，用第五章的 `@tool` 写一个本地工具。合并后的列表就是 `create_agent(tools=...)` 的参数。

MCP 工具的 `metadata` 中有 `mcp` 键，本地工具没有设置元数据，可以据此区分来源。

In [25]:
@tool
def multiply(a: int, b: int) -> int:
    """计算两个整数之积。"""
    return a * b


agent_tools = [*mcp_tools, multiply]


def tool_source(item) -> str:
    """根据元数据判断工具来自 MCP 服务还是本地函数。"""
    return "MCP" if item.metadata and "mcp" in item.metadata else "本地"


for item in agent_tools:
    rprint(item)
    print(f"{item.name:<16} 来源: {tool_source(item):<4} 类型: {type(item).__name__}")

tool_names = [item.name for item in agent_tools]
print("工具名互不相同:", len(tool_names) == len(set(tool_names)))

StructuredTool(
    name='calc_add',
    description='计算两个整数之和。',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}},
        'required': ['a', 'b']
    },
    metadata={
        'mcp': {
            'tool': {'_meta': {'fastmcp': {'tags': []}}},
            'server': {'name': 'MCPRouter', 'version': '4.1.0'}
        }
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x11759a700>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x11cfa1bc0>
)

calc_add         来源: MCP  类型: StructuredTool


StructuredTool(
    name='calc_divide',
    description='计算分子除以分母，并按指定精度四舍五入。',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'numerator': {'type': 'number'},
            'denominator': {'type': 'number'},
            'precision': {
                'default': 2,
                'description': '保留的小数位数，范围为 0 到 6。',
                'maximum': 6,
                'minimum': 0,
                'type': 'integer'
            }
        },
        'required': ['numerator', 'denominator']
    },
    metadata={
        'mcp': {
            'tool': {'_meta': {'fastmcp': {'tags': []}}},
            'server': {'name': 'MCPRouter', 'version': '4.1.0'}
        }
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x11759a700>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x11cfa1da0>
)

calc_divide      来源: MCP  类型: StructuredTool


StructuredTool(
    name='plans_get_plan',
    description='查询一个套餐的价格、成员人数上限、月度 AI 问答额度和 API 调用额度。',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {'plan': {'enum': ['试用版', '基础版', '专业版', '企业版'], 'type': 'string'}},
        'required': ['plan']
    },
    metadata={
        'mcp': {
            'tool': {'_meta': {'fastmcp': {'tags': []}}},
            'server': {'name': 'MCPRouter', 'version': '4.1.0'}
        }
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x11759a700>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x11cfa2ca0>
)

plans_get_plan   来源: MCP  类型: StructuredTool


StructuredTool(
    name='multiply',
    description='计算两个整数之积。',
    args_schema=<class 'langchain_core.utils.pydantic.multiply'>,
    func=<function multiply at 0x11cdc3380>
)

multiply         来源: 本地   类型: StructuredTool
工具名互不相同: True


四个工具都是 `StructuredTool`，对 Agent 来说用法相同；区别只在执行时：MCP 工具通过 Client 请求服务，本地工具直接调用函数。

### 6.4.2 发起一次需要三个来源的问答

问题需要查询套餐价格（套餐服务）、计算总价（本地乘法）和每人每天的费用（计算服务的除法）。系统提示要求所有计算都调用工具，便于观察三个来源。

下面开始请求模型。每次执行 MCP 工具时，Adapter 都会重新连接配置中的服务（见第一节 1.4 的补充）。

In [26]:
SYSTEM_PROMPT = (
    "你是 atguigu助手的客服。套餐信息必须调用套餐工具查询；"
    "所有计算都调用计算工具，不要心算。用中文简短回答。"
)
MIXED_QUESTION = "专业版 12 个人用一个月一共多少钱？按 30 天算，每人每天大约多少钱？保留 2 位小数。"

mixed_agent = create_agent(model=model, tools=agent_tools, system_prompt=SYSTEM_PROMPT)
mixed_result = await mixed_agent.ainvoke(
    {"messages": [{"role": "user", "content": MIXED_QUESTION}]},
    config={"recursion_limit": 12},
)
print("消息条数:", len(mixed_result["messages"]))

消息条数: 10


### 6.4.3 从消息记录看工具来源

按顺序打印模型提出的工具调用、工具结果和最终回答，并标出每个工具的来源。

In [27]:
sources = {item.name: tool_source(item) for item in agent_tools}

for message in mixed_result["messages"]:
    if isinstance(message, HumanMessage):
        print("[HumanMessage]", message.text)
    elif isinstance(message, AIMessage) and message.tool_calls:
        for call in message.tool_calls:
            print(f"[AIMessage] 请求 {call['name']}（{sources.get(call['name'], '未知')}）",
                  json.dumps(call["args"], ensure_ascii=False))
    elif isinstance(message, ToolMessage):
        print(f"[ToolMessage] {message.name} status={message.status} 结果: {message.text}")
    elif isinstance(message, AIMessage):
        print("[AIMessage] 回答:", message.text)

call_ids = [call["id"] for message in mixed_result["messages"]
            if isinstance(message, AIMessage) for call in message.tool_calls]
result_ids = [message.tool_call_id for message in mixed_result["messages"]
              if isinstance(message, ToolMessage)]
print()
print("调用 ID 与工具结果一一对应:", sorted(call_ids) == sorted(result_ids))
print("用到的来源:", sorted({sources.get(name, "未知") for name in
                         [call["name"] for message in mixed_result["messages"]
                          if isinstance(message, AIMessage) for call in message.tool_calls]}))

[HumanMessage] 专业版 12 个人用一个月一共多少钱？按 30 天算，每人每天大约多少钱？保留 2 位小数。
[AIMessage] 请求 plans_get_plan（MCP） {"plan": "专业版"}
[ToolMessage] plans_get_plan status=success 结果: {"套餐":"专业版","价格":"199 元 / 用户 / 月","成员人数上限":"50 人","月度AI问答额度":"30000 次","API调用额度":"每月 80000 次"}
[AIMessage] 请求 multiply（本地） {"a": 199, "b": 12}
[ToolMessage] multiply status=success 结果: 2388
[AIMessage] 请求 multiply（本地） {"a": 12, "b": 30}
[ToolMessage] multiply status=success 结果: 360
[AIMessage] 请求 calc_divide（MCP） {"numerator": 2388, "denominator": 360, "precision": 2}
[ToolMessage] calc_divide status=success 结果: 6.63
[AIMessage] 回答: 专业版 199 元/用户/月，12 人一个月共 **2388 元**。
按 30 天算（12×30=360 人·天），每人每天约 **6.63 元**。

调用 ID 与工具结果一一对应: True
用到的来源: ['MCP', '本地']


模型按工具名选择工具，不关心它来自哪里；工具执行后都返回 `ToolMessage`，调用 ID 一一对应。模型一次回复中可能同时提出多个工具调用，实际顺序与次数以输出为准。

## 6.5 连接失败发生在哪里

![同一个故障，在不同位置表现不同](assets/19-mcp-connection-failures.svg)

LangChain 官方文档的说明是：服务端报告的错误会成为 `status="error"` 的 ToolMessage，交给模型处理；**传输层的失败仍会抛出异常**，因为模型无法处理它们。下面让套餐服务在两个时机连不上，并分别用配置字典和单服务连接观察。

### 6.5.1 准备辅助函数

为了能在实验中途停止套餐服务，把它改为 HTTP 运行。`running_plan_http_server` 与第四节 4.5 的启动器作用相同：进入时启动服务并等待端口可连接，退出时停止服务；这里不保存服务日志。

`capture_fastmcp_logs` 暂时把 FastMCP 的日志收集到列表中，不在输出里打印完整的错误堆栈，用 `summarize_logs` 只看每条日志的级别和第一行。`try_tool` 用手工 ToolCall 执行一个工具，返回 ToolMessage 的状态，或抛出的异常。这些都是本节编写的辅助函数，不是 FastMCP 或 LangChain 的接口。

In [28]:
def choose_available_port(host: str = "127.0.0.1") -> int:
    """让操作系统分配一个本机可用端口，取得编号后关闭探测 socket。"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        probe.bind((host, 0))
        return probe.getsockname()[1]


async def stop_server_process(process) -> int:
    """请求停止服务并等待退出，超过 10 秒则强制停止。"""
    if process.returncode is None:
        process.terminate()
    try:
        return await asyncio.wait_for(process.wait(), timeout=10)
    except asyncio.TimeoutError:
        process.kill()
        return await process.wait()


@asynccontextmanager
async def running_plan_http_server(host: str = "127.0.0.1", path: str = "/mcp"):
    port = choose_available_port(host)
    process = await asyncio.create_subprocess_exec(
        sys.executable, str(PLAN_SCRIPT),
        "--transport", "http", "--host", host, "--port", str(port), "--path", path,
        cwd=str(PROJECT_ROOT),
        stdout=asyncio.subprocess.DEVNULL,
        stderr=asyncio.subprocess.DEVNULL,
    )
    try:
        for _ in range(300):
            if process.returncode is not None:
                raise RuntimeError("套餐服务启动失败")
            try:
                _, writer = await asyncio.open_connection(host, port)
            except OSError:
                await asyncio.sleep(0.05)
            else:
                writer.close()
                await writer.wait_closed()
                break
        else:
            raise TimeoutError("等待套餐服务监听超时")
        yield {"url": f"http://{host}:{port}{path}", "process": process}
    finally:
        await stop_server_process(process)


@contextmanager
def capture_fastmcp_logs():
    """暂时把 FastMCP 的日志收集到列表中，退出时恢复原来的输出方式。"""
    fastmcp_logger = logging.getLogger("fastmcp")
    records = []

    class ListHandler(logging.Handler):
        def emit(self, record):
            records.append(record)

    original_handlers = fastmcp_logger.handlers[:]
    fastmcp_logger.handlers = [ListHandler()]
    try:
        yield records
    finally:
        fastmcp_logger.handlers = original_handlers


def summarize_logs(records) -> None:
    warnings = [record for record in records if record.levelno >= logging.WARNING]
    if not warnings:
        print("  （没有 WARNING 及以上的日志）")
    for record in warnings:
        error_type = type(record.exc_info[1]).__name__ if record.exc_info else None
        print(f"  {record.levelname}: {record.getMessage().splitlines()[0]}（异常类型: {error_type}）")


async def try_tool(tool_obj, args: dict, call_id: str) -> str:
    """手工填写 ToolCall 执行工具，返回 ToolMessage 的状态，或抛出的异常。"""
    try:
        message = await tool_obj.ainvoke(
            {"name": tool_obj.name, "args": args, "id": call_id, "type": "tool_call"}
        )
    except Exception as exc:
        return f"抛出 {type(exc).__name__}: {exc}"
    return f"ToolMessage status={message.status}: {message.text[:60]}"

### 6.5.2 发现工具时：服务已经连不上

配置字典中，计算服务正常；套餐服务指向一个没有服务监听的 URL，模拟服务没有启动。先预测：`list_tools()` 会抛出异常吗？

In [29]:
dead_url = f"http://127.0.0.1:{choose_available_port()}/mcp"
startup_config = {
    "mcpServers": {
        "calc": stdio_server(CALC_SCRIPT),
        "plans": {"url": dead_url},
    }
}
# 多服务连接时，如果有服务连接上了，就不会报错
with capture_fastmcp_logs() as startup_logs:
    async with MCPAdapter(startup_config) as adapter:
        startup_tools = await adapter.list_tools()

print("配置字典发现的工具:", [item.name for item in startup_tools])
print("FastMCP 日志:")
summarize_logs(startup_logs)

配置字典发现的工具: ['calc_add', 'calc_divide']
FastMCP 日志:


`list_tools()` 没有抛出异常：FastMCP 跳过了连不上的服务，只记录一条 WARNING 日志，工具列表里少了 `plans_get_plan`。如果直接创建 Agent，模型根本看不到套餐工具，可能直接凭印象回答价格。

所以使用配置字典时，**发现工具后要检查预期的工具是否齐全**，缺少时在创建 Agent 前停止。对照单服务连接：同一个 URL 直接交给 `MCPAdapter`。

但是如果我们想强制抛出异常，可以使用 `ClientGroup`

```python
from fastmcp import Client, ClientGroup
from langchain.mcp import MCPAdapter

group = ClientGroup.from_config(mcp_config)   # 每个服务一个独立 Client

async with MCPAdapter(group) as adapter:      # MCPAdapter 支持直接传 ClientGroup
    tools = await adapter.list_tools()        # 任一服务连不上，这里就抛 RuntimeError
```

In [30]:
EXPECTED_TOOLS = {"calc_add", "calc_divide", "plans_get_plan"}
missing_tools = EXPECTED_TOOLS - {item.name for item in startup_tools}
print("缺少的工具:", sorted(missing_tools))
if missing_tools:
    print("检查结果: 应在创建 Agent 前停止，先排查对应的服务。")
print()

with capture_fastmcp_logs():
    try:
        async with MCPAdapter(dead_url) as adapter:
            await adapter.list_tools()
    except RuntimeError as exc:
        print("单服务连接:", type(exc).__name__, exc)

缺少的工具: ['plans_get_plan']
检查结果: 应在创建 Agent 前停止，先排查对应的服务。



Traceback (most recent call last):
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3821, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/14/yh05p4nj2z70mjs568rzs9gh0000gn/T/ipykernel_55898/3190378853.py", line 10, in <module>
    async with MCPAdapter(dead_url) as adapter:
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/langchain/mcp/adapter.py", line 209, in __aenter__
    await self._client.__aenter__()
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/client/client.py", line 951, in __aenter__
    return await self._connect()
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/client/client.py", line 1047, in _connect
    raise exception
  File "/Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/cl

单服务连接在 `list_tools()` 时直接抛出 `RuntimeError`，问题当场暴露。

### 6.5.3 执行工具时：发现工具后服务停止

这次先启动套餐 HTTP 服务，用配置字典（计算服务 stdio + 套餐服务 HTTP）和单服务连接各发现一次工具。确认工具可用后停止服务，再执行同样的调用。

同时加一个对照：服务运行时让 `calc_divide` 除以 0，确认业务错误在配置字典下仍是错误消息。

In [ ]:
async with running_plan_http_server() as plan_http:
    runtime_config = {
        "mcpServers": {
            "calc": stdio_server(CALC_SCRIPT),
            "plans": {"url": plan_http["url"]},
        }
    }
    async with MCPAdapter(runtime_config) as adapter:
        config_tools = {item.name: item for item in await adapter.list_tools()}
    async with MCPAdapter(plan_http["url"]) as adapter:
        single_tools = {item.name: item for item in await adapter.list_tools()}
    print("配置字典的工具:", list(config_tools))
    print("单服务连接的工具:", list(single_tools))

    print("服务运行时:")
    print("  配置字典 plans_get_plan →",
          await try_tool(config_tools["plans_get_plan"], {"plan": "基础版"}, "before_config"))
    print("  单服务   get_plan       →",
          await try_tool(single_tools["get_plan"], {"plan": "基础版"}, "before_single"))
    print("  业务错误 calc_divide    →",
          await try_tool(config_tools["calc_divide"], {"numerator": 1, "denominator": 0}, "zero"))

    print("停止套餐服务，退出码:", await stop_server_process(plan_http["process"]))

    with capture_fastmcp_logs() as runtime_logs:
        print("服务停止后:")
        print("  配置字典 plans_get_plan →",
              await try_tool(config_tools["plans_get_plan"], {"plan": "基础版"}, "after_config"))
        print("  配置字典 calc_add       →",
              await try_tool(config_tools["calc_add"], {"a": 2, "b": 3}, "after_calc"))
        print("  单服务   get_plan       →",
              await try_tool(single_tools["get_plan"], {"plan": "基础版"}, "after_single"))

print("FastMCP 日志:")
summarize_logs(runtime_logs)
dead_plan_tool = single_tools["get_plan"]

配置字典的工具: ['calc_add', 'calc_divide', 'plans_get_plan']
单服务连接的工具: ['get_plan']
服务运行时:
  配置字典 plans_get_plan → ToolMessage status=success: {"套餐":"基础版","价格":"99 元 / 用户 / 月","成员人数上限":"10 人","月度AI问答额度":
  单服务   get_plan       → ToolMessage status=success: {"套餐":"基础版","价格":"99 元 / 用户 / 月","成员人数上限":"10 人","月度AI问答额度":
  业务错误 calc_divide    → ToolMessage status=error: Error calling tool 'divide': 除数不能为 0
停止套餐服务，退出码: -15
服务停止后:
  配置字典 plans_get_plan → ToolMessage status=error: Unknown tool: 'plans_get_plan'
  配置字典 calc_add       → ToolMessage status=success: 5
  单服务   get_plan       → 抛出 MCPError: Server returned an error response
FastMCP 日志:


服务停止后，两种连接的表现不同：

| 连接方式 | 发现工具时连不上 | 执行工具时连不上 |
|---|---|---|
| 配置字典 | 不抛异常，跳过该服务，工具列表变少 | 返回 `status="error"` 的 ToolMessage，内容是 `Unknown tool`；计算服务的工具不受影响 |
| 单服务连接 | `list_tools()` 抛出 `RuntimeError` | 工具执行抛出 `RuntimeError` |

原因在于**错误在哪一层被发现**：

- 配置字典的组合服务运行在客户端进程内。每次执行工具都会重新组合，连不上的服务被跳过，组合服务找不到这个工具，就像服务端一样报告一个工具错误。Agent 不会中断，但错误内容没有说明是连接问题。日志中的两条 WARNING 分别来自服务停止后的两次配置字典调用，连计算工具那次也重新组合并跳过了套餐服务。
- 单服务连接中，Client 自己连不上服务，属于传输层失败，只能抛出异常。

`dead_plan_tool` 保存了单服务连接的 `get_plan`，它指向已经停止的服务，下一节用它观察 Agent 的表现。

### 补充：ClientGroup的表现

第一节提到的 `ClientGroup` 也能接入多个服务，工具名同样是 `服务名_工具名`，并保留各服务的真实名称。它进入连接时要求组内所有服务都连上，所以在当前环境中：发现工具时任一服务连不上，`list_tools()` 就抛出 `RuntimeError`；执行工具时某个服务停止，同组其他服务的工具调用也会抛出 `RuntimeError`。它适合要求“所有服务都必须可用”的场景。

## 6.6 用中间件处理连接异常

### 6.6.1 不加中间件：Agent运行中断

把 `dead_plan_tool` 交给 Agent，询问专业版的价格。模型会先提出工具调用，工具执行时抛出异常。

In [31]:
from mcp import MCPError


PLAN_PROMPT = "你是 atguigu助手的客服。套餐信息必须调用 get_plan 查询，用中文简短回答。"
FAILURE_QUESTION = "专业版每月多少钱？"

plain_agent = create_agent(model=model, tools=[dead_plan_tool], system_prompt=PLAN_PROMPT)
try:
    with capture_fastmcp_logs():
        await plain_agent.ainvoke(
            {"messages": [{"role": "user", "content": FAILURE_QUESTION}]},
            config={"recursion_limit": 6},
        )
except MCPError as exc:
    print("Agent 运行中断:", type(exc).__name__, exc)
else:
    print("Agent 正常结束：模型没有调用工具，请重新运行本单元格。")

Agent 运行中断: MCPError Server returned an error response


第八章 1.4.5 讲过：`create_agent` 默认只把参数不合法这类错误转成消息，工具执行中抛出的异常会中断整个 Agent。用户拿不到任何回答。

### 6.6.2 加上ToolRetryMiddleware

沿用第八章 06 的 [ToolRetryMiddleware](../Chapter08_Middleware/06-other中间件.ipynb)：连接失败先重试一次，仍然失败时把异常转成 ToolMessage 交给模型。

| 参数 | 本节取值与原因 |
|---|---|
| `max_retries=1` | 服务可能只是短暂重启，重试一次；总共最多执行 2 次 |
| `tools=["get_plan"]` | 只对套餐工具生效 |
| `retry_on=(RuntimeError,)` | FastMCP 把连接失败包装成 `RuntimeError`；配合 `tools` 限定范围，避免把其他工具的 RuntimeError 也当作连接问题 |
| `initial_delay=0.5`、`jitter=False` | 缩短演示的等待时间，并让等待时间固定 |
| `on_failure=函数` | 重试用完后，用函数返回的文字作为 ToolMessage 内容，不把原始异常信息交给模型 |

In [33]:
def connection_failure_message(exc: Exception) -> str:
    """重试用完后返回给模型的说明，不包含原始异常信息。"""
    return "套餐服务暂时无法连接。请告诉用户稍后再试，或联系人工客服。"


connection_retry = ToolRetryMiddleware(
    max_retries=1,
    tools=["get_plan"],
    retry_on=(RuntimeError,MCPError,),
    initial_delay=0.5,
    jitter=False,
    on_failure=connection_failure_message,
)
guarded_agent = create_agent(
    model=model,
    tools=[dead_plan_tool],
    system_prompt=PLAN_PROMPT,
    middleware=[connection_retry],
)

with capture_fastmcp_logs():
    guarded_result = await guarded_agent.ainvoke(
        {"messages": [{"role": "user", "content": FAILURE_QUESTION}]},
        config={"recursion_limit": 6},
    )

for message in guarded_result["messages"]:
    if isinstance(message, HumanMessage):
        print("[HumanMessage]", message.text)
    elif isinstance(message, AIMessage) and message.tool_calls:
        for call in message.tool_calls:
            print("[AIMessage] 请求", call["name"], json.dumps(call["args"], ensure_ascii=False))
    elif isinstance(message, ToolMessage):
        print(f"[ToolMessage] {message.name} status={message.status} 内容: {message.text}")
    elif isinstance(message, AIMessage):
        print("[AIMessage] 回答:", message.text)

[HumanMessage] 专业版每月多少钱？
[AIMessage] 请求 get_plan {"plan": "专业版"}
[ToolMessage] get_plan status=error 内容: 套餐服务暂时无法连接。请告诉用户稍后再试，或联系人工客服。
[AIMessage] 回答: 抱歉，套餐服务暂时无法连接，暂时查不到专业版的价格。请您稍后再试，或联系人工客服咨询。


这次 Agent 没有中断：中间件把连接异常转成 `status="error"` 的 ToolMessage，模型据此告诉用户服务暂时不可用，而不是编造价格。

第八章 1.4.5 还介绍过 `ToolErrorMiddleware`：它不重试，`on_error` 返回文字时把异常转成错误消息，返回 None 时照常抛出。需要“先重试、再转消息”时用本节的写法；只需要精确控制哪些异常交给模型时，可以改用它（见小练习）。

## 6.7 小结与小练习

| 情况 | 失败出现在哪里 | 表现 | 处理方式 |
|---|---|---|---|
| 配置字典，发现工具时连不上 | `list_tools()` | 跳过该服务，只有 WARNING 日志 | 检查预期工具是否齐全 |
| 配置字典，执行工具时连不上 | 工具执行 | `status="error"`，`Unknown tool` | 模型能收到，但内容不说明原因 |
| 单服务连接，发现工具时连不上 | `list_tools()` | 抛出 `RuntimeError` | 创建 Agent 前就停止 |
| 单服务连接，执行工具时连不上 | 工具执行 | 抛出 `RuntimeError`，Agent 中断 | 中间件重试，再转成错误消息 |
| 参数错误、业务错误（第三节） | 服务端 | `status="error"` 的 ToolMessage | 模型根据错误内容调整 |

> ⚠️ **与旧版教程的差异**：LangChain 1.4.0 之前使用 `langchain-mcp-adapters` 的 `MultiServerMCPClient`，配置字典没有外层的 `mcpServers`，每个服务要写 `transport`，用 `get_tools()` 获取工具，并可通过 `tool_name_prefix` 选择是否加前缀。当前的 `MCPAdapter` 使用 `mcpServers` 格式，连接方式由字段推断，多服务时自动加上 `服务名_` 前缀。

小练习：

1. 把 6.3.3 的配置键改成 `pricing`，预测套餐工具的新名称，再验证。
2. 把本地工具 `multiply` 改名为 `calc_add`（函数名改成 `calc_add` 即可），和 MCP 工具一起交给 `create_agent`，观察是否报错。再让 Agent 计算一次加法，根据 ToolMessage 的 `artifact` 判断实际执行的是哪一个工具。
3. 把 6.6.2 的 `on_failure` 改成 `"error"`，预测 Agent 的表现，再验证。
4. 用 `ToolErrorMiddleware` 改写 6.6.2：`on_error` 只在异常是 `RuntimeError` 时返回提示文字，其他异常返回 None。比较它与 `ToolRetryMiddleware` 的消息记录有什么不同。

第十一章到此结束。第十二章学习 Skill，综合实战会继续复用计算服务和套餐服务。

参考：[LangChain MCP Connections](https://docs.langchain.com/oss/python/langchain/mcp/connections)、[LangChain MCP Tools](https://docs.langchain.com/oss/python/langchain/mcp/tools)、[LangChain MCP 迁移指南](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters)、[FastMCP 配置文件格式](https://gofastmcp.com/integrations/mcp-json-configuration)。接口与输出以本节打印的环境版本为准。